<a href="https://colab.research.google.com/github/ananyavj/dental.ai/blob/main/Yolo_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lokisilvres/dental-disease-panoramic-detection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dental-disease-panoramic-detection-dataset' dataset.
Path to dataset files: /kaggle/input/dental-disease-panoramic-detection-dataset


In [2]:
!pip install ultralytics
from ultralytics import YOLO
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
print(os.listdir(path))

['checkpoint0033_4scale.pth', 'COCO', 'best.pt', 'YOLO']


In [4]:
# Check what's inside the YOLO folder
yolo_path = os.path.join(path, 'YOLO')
print(os.listdir(yolo_path))

['YOLO']


In [5]:
import shutil

# This copies the YOLO folder to your main Colab area so you can use it easily
shutil.copytree(yolo_path, '/content/dental_project')

print("Files moved to /content/dental_project")


Files moved to /content/dental_project


In [6]:
print(os.listdir("/content/dental_project/YOLO"))

['test', 'data.yaml', 'train', 'valid']


In [ ]:
from ultralytics import YOLO
import os
import yaml

# 1. Load the pre-trained model
model = YOLO('yolov11n.pt')

# Path to the data.yaml file
data_yaml_path = '/content/dental_project/YOLO/data.yaml'

# Read the content of data.yaml
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Update the paths to reflect the current Colab environment
# The actual image folders are located at /content/dental_project/YOLO/{train, valid, test}/images
data_config['train'] = '/content/dental_project/YOLO/train/images'
data_config['val'] = '/content/dental_project/YOLO/valid/images'
data_config['test'] = '/content/dental_project/YOLO/test/images'

# Write the modified content back to data.yaml
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"Updated {data_yaml_path} with correct image paths.")

# 2. Train!
# NOTE: Check if the file is actually named 'data.yaml' or 'coco.yaml'
# inside the dental_project folder and change the name below if needed.
model.train(
    data=data_yaml_path, # Use the updated path
    epochs=25,
    imgsz=640,
    device='cpu',  # Changed to 'cpu' as no GPU is available
    save=True,
)

Updated /content/dental_project/YOLO/data.yaml with correct image paths.
Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dental_project/YOLO/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opse

In [ ]:
def get_fdi_name(fdi_code):
    """Returns the anatomical name of a tooth based on FDI code."""
    fdi_code = str(fdi_code)
    quadrant = fdi_code[0]
    position = fdi_code[1]

    names = {
        '1': "Central Incisor", '2': "Lateral Incisor", '3': "Canine",
        '4': "1st Premolar", '5': "2nd Premolar", '6': "1st Molar",
        '7': "2nd Molar", '8': "3rd Molar"
    }

    quadrants = {
        '1': "Upper Right", '2': "Upper Left",
        '3': "Lower Left", '4': "Lower Right"
    }

    return f"{quadrants.get(quadrant)} {names.get(position)}"

# Example:
print(get_fdi_name(16)) # Output: Upper Right 1st Molar

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Evaluate the model on the validation set
metrics = model.val(
    data=data_yaml_path,
    imgsz=640,
    device=0  # Use the GPU for evaluation
)

print("Validation metrics:", metrics)

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

# Load your best-trained model
best_model = YOLO('/content/runs/detect/train/weights/best.pt')

# Run prediction on a new X-ray image
results = best_model.predict(source='content/dental_project/YOLO/test/images', conf=0.25)

for r in results:
    im_array = r.plot() # Plot the boxes on the image
    cv2_imshow(im_array)

    for box in r.boxes:
        cls_id = int(box.cls[0])
        label = best_model.names[cls_id]

        if label.isdigit(): # If the label is an FDI number
            print(f"Detected Tooth: FDI {label}")
        elif "caries" in label.lower() or "decay" in label.lower():
            print(f"WARNING: Caries detected!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get class names from the data_config
class_names = data_config['names']

# Best performing classes identified from the validation output (for informational purposes)
best_classes = ['Implant', 'impacted tooth', 'Crown']

plt.figure(figsize=(10, 8))
plt.title('Overall Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)

# Get the overall precision and recall arrays from metrics.box
# metrics.box.p and metrics.box.r contain the aggregated overall precision and recall values
p_overall = metrics.box.p
r_overall = metrics.box.r

# Plot the overall PR curve
plt.plot(r_overall, p_overall, label=f'Overall (mAP50: {metrics.box.map50:.3f})')

plt.legend()
plt.show()